# Analyze Local Suno Audio Batches

This notebook works only with the files already stored in `data/suno-audio`.

It will:
- find which `batch_*` folders are complete enough to load,
- count the total number of locally available rows,
- count normalized comma-separated tags from the `tags` column,
- save the tag counts to `genre_counts.csv`.

Important:
- the dataset `tags` field mixes genres, moods, instruments, and styles,
- the counts below are therefore tag frequencies, not a strict one-genre-per-song summary,
- if some batches are incomplete, the totals reflect only the successfully loaded local batches.

In [2]:
from collections import Counter
from pathlib import Path
import csv

import pandas as pd
from datasets import load_from_disk

/Library/Python/3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Python/3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [1]:
DATA_ROOT = Path.cwd() / "data" / "suno-audio"
OUTPUT_CSV = Path.cwd() / "genre_counts.csv"

if not DATA_ROOT.exists():
    raise FileNotFoundError(f"Could not find local dataset folder: {DATA_ROOT}")

print(f"Data root: {DATA_ROOT}")
print(f"Output CSV: {OUTPUT_CSV}")

NameError: name 'Path' is not defined

In [40]:
loadable_batches = []
failed_batches = []

for batch_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir() and p.name.startswith("batch_")):
    try:
        ds = load_from_disk(str(batch_dir))
        loadable_batches.append({"batch": batch_dir.name, "rows": ds.num_rows})
    except Exception as exc:
        failed_batches.append({"batch": batch_dir.name, "error": str(exc)})

loadable_df = pd.DataFrame(loadable_batches)
failed_df = pd.DataFrame(failed_batches)

print(f"Loadable batches: {len(loadable_df)}")
print(f"Failed batches: {len(failed_df)}")
display(loadable_df)

if not failed_df.empty:
    display(failed_df.head(10))

Loadable batches: 15
Failed batches: 35


,batch,rows
0,batch_0,995
1,batch_1,992
2,batch_10,995
3,batch_11,992
4,batch_12,990
5,batch_13,996
6,batch_14,994
7,batch_15,993
8,batch_16,994
9,batch_17,997


,batch,error
0,batch_22,[Errno 2] No such file or directory: '/Users/m...
1,batch_23,Directory /Users/mingoosim/Desktop/DS340/Proej...
2,batch_24,Directory /Users/mingoosim/Desktop/DS340/Proej...
3,batch_25,Directory /Users/mingoosim/Desktop/DS340/Proej...
4,batch_26,Directory /Users/mingoosim/Desktop/DS340/Proej...
5,batch_27,Directory /Users/mingoosim/Desktop/DS340/Proej...
6,batch_28,Directory /Users/mingoosim/Desktop/DS340/Proej...
7,batch_29,Directory /Users/mingoosim/Desktop/DS340/Proej...
8,batch_3,Directory /Users/mingoosim/Desktop/DS340/Proej...
9,batch_30,Directory /Users/mingoosim/Desktop/DS340/Proej...


## Count rows and tags

Normalization used below:
- lower-case everything,
- split on commas,
- trim surrounding whitespace,
- collapse repeated internal spaces,
- drop empty tags.

In [41]:
total_rows = 0
tag_counter = Counter()

for row in loadable_batches:
    batch_dir = DATA_ROOT / row["batch"]
    ds = load_from_disk(str(batch_dir))
    total_rows += ds.num_rows

    tags_column = ds.select_columns(["tags"])["tags"]
    for raw_tags in tags_column:
        if raw_tags is None:
            continue

        for part in str(raw_tags).split(","):
            tag = " ".join(part.strip().lower().split())
            if tag:
                tag_counter[tag] += 1

tag_counts_df = pd.DataFrame(tag_counter.most_common(), columns=["genre_tag", "count"])

print(f"Total local rows: {total_rows}")
print(f"Unique normalized tags: {len(tag_counts_df)}")
display(tag_counts_df.head(50))

Total local rows: 14907
Unique normalized tags: 14296


,genre_tag,count
0,pop,703
1,rap,612
2,rock,501
3,electro,329
4,piano,307
5,violin,299
6,trap,283
7,female vocals,282
8,dark,271
9,metal,245


In [42]:
tag_counts_df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved tag counts to: {OUTPUT_CSV}")

Saved tag counts to: /Users/mingoosim/Desktop/DS340/Proejct/genre_counts.csv


In [43]:
TOP_N = 25
tag_counts_df.head(TOP_N)

,genre_tag,count
0,pop,703
1,rap,612
2,rock,501
3,electro,329
4,piano,307
5,violin,299
6,trap,283
7,female vocals,282
8,dark,271
9,metal,245
